In [1]:
import cv2
import numpy as np
import tensorflow as tf
# from tflite_runtime.interpreter import Interpreter
import glob
import os

# model_path = "./model.tflite"
# interpreter = Interpreter(model_path=model_path, num_threads=4)
# interpreter.allocate_tensors()

# input_details = interpreter.get_input_details()
# output_details = interpreter.get_output_details()

model = tf.keras.models.load_model("./model.h5", compile=False)

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the TFLite model
with open("model.tflite", "wb") as f:
    f.write(tflite_model)

I0000 00:00:1762977090.804350  198307 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1762977090.831030  198307 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1762977091.427689  198307 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1762977091.916096  198307 gpu_device.cc:2456] TensorFlow was not built with CUDA kernel bi

INFO:tensorflow:Assets written to: /tmp/tmpppx7l5fa/assets


INFO:tensorflow:Assets written to: /tmp/tmpppx7l5fa/assets


Saved artifact at '/tmp/tmpppx7l5fa'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 256, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 256, 256, 4), dtype=tf.float32, name=None)
Captures:
  127045405179248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127045403283824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127045403288576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127045403290688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127045403293680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127045403294384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127045403295440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127045403294560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127045403295088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127045403380368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  12704

W0000 00:00:1762977092.922680  198307 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1762977092.922695  198307 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1762977092.922901  198307 reader.cc:83] Reading SavedModel from: /tmp/tmpppx7l5fa
I0000 00:00:1762977092.923585  198307 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1762977092.923590  198307 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpppx7l5fa
I0000 00:00:1762977092.931845  198307 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1762977092.933190  198307 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1762977092.993704  198307 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpppx7l5fa
I0000 00:00:1762977093.008577  198307 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 85682 microseconds.
I0000 00:00:1762977093.026370  198307

In [5]:
CLASS_COLORS = {
    1: [255,   0,   0],   # ship -> red
    2: [128, 128, 128],   # sargassum -> gray
    3: [255, 255, 255],   # oil -> white
}

# Open video
cap = cv2.VideoCapture("../video/ManchaSat.mp4")

def preprocess_frame(frame):
    frame = cv2.resize(frame, (256, 256))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = gray[..., np.newaxis]
    return np.expand_dims(gray.astype("float32") / 255.0, axis=0)

def decode_mask(pred, frame_shape):
    mask = np.argmax(pred[0], axis=-1).astype(np.uint8)
    mask = cv2.resize(mask, (frame_shape[1], frame_shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask

def colorize_mask(mask):
    # Create a 3-channel RGB canvas
    mask_rgb = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)

    # Ships = class 1 -> red
    mask_rgb[mask == 1] = (0, 0, 255)

    # Oil = class 6 -> white
    mask_rgb[mask == 3] = (255, 255, 255)

    # Sargassum = class 7 -> orange (BGR)
    mask_rgb[mask == 2] = (0, 165, 255)

    return mask_rgb

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    input_tensor = preprocess_frame(frame)
    input_index = input_details[0]['index']
    interpreter.set_tensor(input_index, input_tensor)
    interpreter.invoke()
    output_index = output_details[0]['index']
    pred = interpreter.get_tensor(output_index)

    mask = decode_mask(pred, frame.shape)
    mask_colored = colorize_mask(mask)


    oil_pixels = np.sum(mask == 3)
    total_pixels = mask.size

    cv2.putText(
        mask_colored,
        f"Area: {(oil_pixels/total_pixels)*100:.2f}%",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        1,
        cv2.LINE_AA,
    )

    # Show only the mask (not the original frame)
    cv2.imshow("Segmentation Mask", mask_colored)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
image_folder = "../grayscale_frames"
mask_folder = "../mask_frames"

# Load files
image_files = sorted(glob.glob(os.path.join(image_folder, "*.png")))
mask_files = sorted(glob.glob(os.path.join(mask_folder, "*.png")))

def preprocess_frame(frame):
    frame = cv2.resize(frame, (256, 256))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = gray[..., np.newaxis]
    return np.expand_dims(gray.astype("float32") / 255.0, axis=0)

def decode_mask(pred, frame_shape):
    mask = np.argmax(pred[0], axis=-1).astype(np.uint8)
    mask = cv2.resize(mask, (frame_shape[1], frame_shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask

# Counters
pred_total = 0
gt_total = 0
iou_total = 0
dice_total = 0
num_images = min(len(image_files), len(mask_files))

for img_path, mask_path in zip(image_files, mask_files):
    # Load input image and predict
    frame = cv2.imread(img_path)
    input_tensor = preprocess_frame(frame)
    pred = model.predict(input_tensor, verbose=0)
    pred_mask = decode_mask(pred, frame.shape)

    # Predicted oil mask (class 3)
    pred_oil = (pred_mask == 3).astype(np.uint8)

    # Ground-truth oil mask (assuming white=oil)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = cv2.resize(gt_mask, (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST)
    gt_oil = (gt_mask > 127).astype(np.uint8)

    # Pixel counts
    pred_total += np.sum(pred_oil)
    gt_total += np.sum(gt_oil)

    # Intersection and Union
    intersection = np.sum((pred_oil & gt_oil) == 1)
    union = np.sum((pred_oil | gt_oil) == 1)

    # IoU
    iou = intersection / union if union > 0 else 0
    iou_total += iou

    # Dice score = 2 * intersection / (pred + gt)
    denom = np.sum(pred_oil) + np.sum(gt_oil)
    dice = (2 * intersection / denom) if denom > 0 else 0
    dice_total += dice

# Averages
avg_pred = pred_total / num_images
avg_gt = gt_total / num_images
avg_iou = iou_total / num_images
avg_dice = dice_total / num_images

percent_diff = abs(avg_pred - avg_gt) / avg_gt * 100 if avg_gt > 0 else 0

print(f"Average predicted oil pixels: {avg_pred:.2f}")
print(f"Average ground-truth oil pixels: {avg_gt:.2f}")
print(f"Difference: {avg_pred - avg_gt:.2f} px")
print(f"Percent difference: {percent_diff:.2f}%")
print(f"Average IoU: {avg_iou:.3f}")
print(f"Average Dice: {avg_dice:.3f}")